In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer
import cassio
from langchain.vectorstores.cassandra import Cassandra
from langchain.embeddings import HuggingFaceBgeEmbeddings

e:\qna_chatbot\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_CROP_API")
ASTRA_DB_ID = os.getenv("ASTRA_DB_CROP_ID")

In [3]:
# 1. Load the CSV
df = pd.read_csv("Crop_recommendation.csv")

In [4]:
# 4. Initialize Astra DB connection
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

DriverException: Unable to connect to the metadata service at https://cfe95760-fbf8-43ad-bcb6-923e56d2fe3f-us-east-2.db.astra.datastax.com:29080/metadata. Check the cluster status in the cloud console. 

In [5]:
# 5. Create embedding model (using Hugging Face)
embedding = HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\Anurup Datta\AppData\Local\Temp\ipykernel_39964\2326891156.py:2: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 639.66it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical

In [6]:
# 6. Create Cassandra vector store
astra_vector_store = Cassandra(
    embedding=embedding,
    table_name="crop_vectors",
    session=None,  # CassIO manages the session
    keyspace=None, # CassIO manages the keyspace
)

In [7]:
# 7. Prepare crop strings for embedding and storage
crop_strings = df.apply(
    lambda row: f"N {row['N']} P {row['P']} K {row['K']} temperature {row['temperature']} humidity {row['humidity']} ph {row['ph']} rainfall {row['rainfall']} label {row['label']}",
    axis=1
).tolist()

# 8. Store crop embeddings in Astra DB (only needs to be done once; skip if already stored)
astra_vector_store.add_texts(crop_strings)

['db3aafee00cb46eb9be96a1014626cab',
 '98657515ea694c36ad884043c8cba9ca',
 'd709f012030248439b8fc5c438096822',
 '58feee0651bb42a5b81ad0fb79363e26',
 '45fcd2364171477787958bb41d6c5633',
 '39822d49a7e04585beac5718f5d14961',
 'abcd18ad65414a3d8d6cd334ae54b17a',
 '6ea9e14ccb7a4710b46f3402493a5dfa',
 '0bfbd8056f2b465c90e36d218066b20e',
 '5689d65dc8df411e93c7865b9efccf54',
 '528d7a80adb5454ea5f26fd7d2b785cf',
 '3dde7cbb3ccf4507a6d5540a079e9ece',
 '6667d37085244ccf936de320876fd88e',
 'f24417a172d6425f9b763cf651cccfc3',
 '7485a569486d438981206ed82198c3d7',
 'efe23ab2457f493c853a42102fe570d6',
 '5fb52c1a14724aa186e2e3df335671cb',
 'fe7707af3b8b44eb8496d13fb251fc7e',
 '1261813005db4c9ba016ed33cfde592a',
 '77e9e477df3e428c9dbb94732c6de56b',
 '6eeb603bf5534fa293e98776b71c801b',
 '7f0653c743cf44beb91098bbb4da8e45',
 '08ce81e8e1a541c39ed9d86e2176f6e5',
 '23c129f3af624d72bc54023d36aef045',
 'c8138269e97e4a8f9ded48f7d50ebd2e',
 'de31f2dc75b34aea887043e0c86b9d85',
 'fd3878cffe994857be93cd25bd6a0267',
 

In [8]:
# 11. Get Groq LLM response
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"  # Use your preferred Groq model
)

In [ ]:
# e:\qna_chatbot\capstone\crop_recommend.ipynb

# --- Previous cells are the same ---

# 2. Input parameters
humidity = float(input("Enter humidity percentage: "))
moisture = float(input("Enter moisture percentage: "))
temperature = float(input("Enter temperature (°C): "))
# Add an optional keyword input for the hybrid search
keyword_query = input("Enter a specific crop or keyword to search for (e.g., 'rice', optional): ")

# 3. Create input string for embedding (for the semantic part)
input_str_semantic = f"humidity {humidity} moisture {moisture} temperature {temperature}"

# 9. Perform Hybrid Search in Astra DB
# The text query goes into the 'query' parameter for the keyword part of the search.
# The embedding for the semantic part is generated automatically from the 'query' text.
# For more control, you can pass an embedding directly.

# The `similarity_search` method in the Cassandra vector store can perform hybrid search
# if you provide a text query that includes keywords. Astra DB's backend will
# handle both the vector and text search.
if keyword_query:
    # Hybrid search: combines keyword and semantic
    search_query = f"{input_str_semantic} {keyword_query}"
    print(f"\nPerforming hybrid search for: '{search_query}'")
else:
    # Pure semantic search
    search_query = input_str_semantic
    print(f"\nPerforming semantic search for: '{search_query}'")

results = astra_vector_store.similarity_search(search_query, k=4)

# The rest of the logic remains the same
top_crops = [doc.page_content.split("label ")[-1] for doc in results]

# 10. Prepare prompt for Groq
crop_list = ", ".join(top_crops)
prompt = (
    f"Given the environmental conditions: humidity {humidity}%, "
    f"moisture {moisture}%, temperature {temperature}°C, "
    f"and a user keyword '{keyword_query}', the top 4 suitable crops are: {crop_list}. "
    "Explain why these crops are suitable, considering both the conditions and the keyword."
)
response = llm.invoke(prompt)

# 12. Display results
print("\nTop 4 suitable crops:", crop_list)
print("Groq LLM explanation:\n", response.content)


Top 4 suitable crops: mungbean, mungbean, mungbean, mungbean
Groq LLM explanation:
 content="The top 4 suitable crops being all mungbean suggests that the given environmental conditions are highly favorable for mungbean cultivation. Here's why:\n\n1. **Temperature**: Mungbeans thrive in warm temperatures, typically between 20°C to 30°C. The given temperature of 25.0°C falls within this optimal range, making it suitable for mungbean growth.\n2. **Humidity**: Mungbeans prefer a relatively low to moderate humidity level, around 40% to 60%. The given humidity of 39.0% is slightly below this range, but still within acceptable limits for mungbean cultivation.\n3. **Moisture**: Mungbeans require adequate moisture, especially during the germination and flowering stages. The given moisture level of 21.0% may seem relatively low, but mungbeans are known to be drought-tolerant and can thrive in areas with moderate moisture levels.\n\nConsidering these factors, the environmental conditions seem to